# MouserCV - Notebook 2: Evaluate SuperAnimal Keypoints

This notebook evaluates **DeepLabCut SuperAnimal-Quadruped** for zero-shot
keypoint detection on mouse behavioral videos.

**Workflow:**
1. Download video from GCS
2. Run DLC SuperAnimal-Quadruped zero-shot inference
3. Analyze per-keypoint reliability
4. Identify the most useful keypoints
5. Export results and upload to GCS

**Runtime:** Select *GPU* (T4 or better) under Runtime > Change runtime type.

## 1. Setup

In [ ]:
!pip install -q "deeplabcut[gui,modelzoo]" google-cloud-storage matplotlib pandas

## 2. Authenticate and Configure

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('Authenticated successfully.')

In [ ]:
# ============================================================
# EDITABLE PARAMETERS
# ============================================================
BUCKET = "mousercv-data"
VIDEO_ID = "test_001"
PROJECT = "demo"
PCUTOFF = 0.3  # Confidence threshold for keypoint reliability
RELIABILITY_THRESHOLD = 0.70  # Fraction above pcutoff to be "reliable"

## 3. Download Video from GCS

In [ ]:
import os
from google.cloud import storage

client = storage.Client()
bucket = client.bucket(BUCKET)

gcs_video_path = f"videos/{PROJECT}/{VIDEO_ID}.mp4"
local_video_path = f"/tmp/{VIDEO_ID}.mp4"

blob = bucket.blob(gcs_video_path)
blob.download_to_filename(local_video_path)
file_size_mb = os.path.getsize(local_video_path) / (1024 * 1024)
print(f"Downloaded {gcs_video_path} -> {local_video_path} ({file_size_mb:.1f} MB)")

## 4. Run DLC SuperAnimal-Quadruped Zero-Shot

This uses the SuperAnimal-Quadruped model with HRNet backbone and
Faster R-CNN detector, plus video adaptation for improved accuracy.

In [ ]:
import deeplabcut

video_path = local_video_path

deeplabcut.video_inference_superanimal(
    [video_path],
    superanimal_name="superanimal_quadruped",
    model_name="hrnet_w32",
    detector_name="fasterrcnn_resnet50_fpn_v2",
    video_adapt=True,
    pcutoff=PCUTOFF,
)
print("SuperAnimal inference complete.")

## 5. Load Results

In [ ]:
import pandas as pd
import glob

# DLC outputs an h5 file next to the video
h5_files = glob.glob(f"/tmp/{VIDEO_ID}*.h5")
if not h5_files:
    # Also check in a DLC-created subdirectory
    h5_files = glob.glob(f"/tmp/*{VIDEO_ID}*/**/*.h5", recursive=True)

if not h5_files:
    raise FileNotFoundError("No h5 output found. Check DLC inference output above.")

h5_path = h5_files[0]
print(f"Loading results from: {h5_path}")

df = pd.read_hdf(h5_path)
print(f"Shape: {df.shape}")
print(f"Columns (first 20): {list(df.columns[:20])}")
print(f"\nMultiIndex levels: {df.columns.names}")
df.head()

## 6. Visualize Keypoint Overlays

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

# Parse the DLC multi-index columns
# Typically: (scorer, individual, bodypart, coord)
scorer = df.columns.get_level_values(0)[0]

# Get unique bodyparts
if df.columns.nlevels == 4:
    # Multi-animal: (scorer, individual, bodypart, coord)
    bodyparts = df.columns.get_level_values(2).unique().tolist()
    individuals = df.columns.get_level_values(1).unique().tolist()
    multi_animal = True
else:
    # Single-animal: (scorer, bodypart, coord)
    bodyparts = df.columns.get_level_values(1).unique().tolist()
    individuals = ["animal"]
    multi_animal = False

print(f"Bodyparts ({len(bodyparts)}): {bodyparts}")
print(f"Individuals: {individuals}")

# Select 5 evenly-spaced frames
total_frames = len(df)
sample_indices = np.linspace(0, total_frames - 1, 5, dtype=int)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
cmap = plt.cm.RdYlGn  # Red=low confidence, Green=high

cap = cv2.VideoCapture(local_video_path)

for plot_idx, frame_idx in enumerate(sample_indices):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    if not ret:
        continue
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    axes[plot_idx].imshow(frame_rgb)

    row = df.iloc[frame_idx]
    for ind in individuals:
        for bp in bodyparts:
            try:
                if multi_animal:
                    x = row[(scorer, ind, bp, "x")]
                    y = row[(scorer, ind, bp, "y")]
                    likelihood = row[(scorer, ind, bp, "likelihood")]
                else:
                    x = row[(scorer, bp, "x")]
                    y = row[(scorer, bp, "y")]
                    likelihood = row[(scorer, bp, "likelihood")]
            except KeyError:
                continue

            if np.isnan(x) or np.isnan(y):
                continue
            color = cmap(likelihood)
            size = 30 if likelihood > PCUTOFF else 10
            axes[plot_idx].scatter(x, y, c=[color], s=size, edgecolors="white",
                                   linewidths=0.5, zorder=5)

    axes[plot_idx].set_title(f"Frame {frame_idx}")
    axes[plot_idx].axis("off")

cap.release()
axes[5].axis("off")
fig.suptitle("SuperAnimal Keypoints (color = confidence: red=low, green=high)",
             fontsize=14)
plt.tight_layout()
plt.show()

## 7. Per-Keypoint Reliability Analysis

For each of the 39 keypoints, compute mean confidence and the percentage
of frames where confidence exceeds the `pcutoff` threshold.

In [ ]:
# Compute per-keypoint statistics
kp_stats = []

for ind in individuals:
    for bp in bodyparts:
        try:
            if multi_animal:
                likelihoods = df[(scorer, ind, bp, "likelihood")].values
            else:
                likelihoods = df[(scorer, bp, "likelihood")].values
        except KeyError:
            continue

        valid = likelihoods[~np.isnan(likelihoods)]
        if len(valid) == 0:
            continue

        mean_conf = float(np.mean(valid))
        pct_above = float(np.mean(valid > PCUTOFF) * 100)
        kp_stats.append({
            "individual": ind,
            "bodypart": bp,
            "mean_confidence": mean_conf,
            "pct_above_pcutoff": pct_above,
            "n_frames": len(valid),
        })

kp_df = pd.DataFrame(kp_stats).sort_values("pct_above_pcutoff", ascending=False)
kp_df = kp_df.reset_index(drop=True)
print(f"Analyzed {len(kp_df)} keypoints across {len(individuals)} individuals\n")
kp_df

In [ ]:
# Bar chart sorted by reliability
fig, ax = plt.subplots(figsize=(16, 8))

labels = [f"{r.individual}/{r.bodypart}" if multi_animal else r.bodypart
          for _, r in kp_df.iterrows()]
colors_bar = ["#2ecc71" if pct >= RELIABILITY_THRESHOLD * 100 else "#e74c3c"
              for pct in kp_df["pct_above_pcutoff"]]

ax.barh(range(len(kp_df)), kp_df["pct_above_pcutoff"], color=colors_bar)
ax.set_yticks(range(len(kp_df)))
ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel("% frames above pcutoff")
ax.set_title(f"Keypoint Reliability (green >= {RELIABILITY_THRESHOLD * 100:.0f}%)")
ax.axvline(x=RELIABILITY_THRESHOLD * 100, color="gray", linestyle="--", alpha=0.7)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 8. Identify Reliable Keypoints

In [ ]:
reliable = kp_df[kp_df["pct_above_pcutoff"] >= RELIABILITY_THRESHOLD * 100]
unreliable = kp_df[kp_df["pct_above_pcutoff"] < RELIABILITY_THRESHOLD * 100]

print(f"RELIABLE keypoints ({len(reliable)}/{len(kp_df)}):")
print("=" * 50)
for _, row in reliable.iterrows():
    label = f"{row.individual}/{row.bodypart}" if multi_animal else row.bodypart
    print(f"  {label:30s}  conf={row.mean_confidence:.3f}  "
          f"above_pcutoff={row.pct_above_pcutoff:.1f}%")

print(f"\nUNRELIABLE keypoints ({len(unreliable)}/{len(kp_df)}):")
print("=" * 50)
for _, row in unreliable.iterrows():
    label = f"{row.individual}/{row.bodypart}" if multi_animal else row.bodypart
    print(f"  {label:30s}  conf={row.mean_confidence:.3f}  "
          f"above_pcutoff={row.pct_above_pcutoff:.1f}%")

## 9. Export to JSONL

In [ ]:
import json

results_dir = f"/tmp/results/{VIDEO_ID}"
os.makedirs(results_dir, exist_ok=True)
keypoints_path = os.path.join(results_dir, "keypoints.jsonl")

with open(keypoints_path, "w") as f:
    for frame_idx in range(len(df)):
        row = df.iloc[frame_idx]
        frame_record = {"frame": frame_idx, "keypoints": {}}

        for ind in individuals:
            ind_kps = {}
            for bp in bodyparts:
                try:
                    if multi_animal:
                        x = float(row[(scorer, ind, bp, "x")])
                        y = float(row[(scorer, ind, bp, "y")])
                        likelihood = float(row[(scorer, ind, bp, "likelihood")])
                    else:
                        x = float(row[(scorer, bp, "x")])
                        y = float(row[(scorer, bp, "y")])
                        likelihood = float(row[(scorer, bp, "likelihood")])
                except (KeyError, TypeError):
                    continue

                if np.isnan(x) or np.isnan(y):
                    continue
                ind_kps[bp] = {
                    "x": round(x, 2),
                    "y": round(y, 2),
                    "likelihood": round(likelihood, 4),
                }
            frame_record["keypoints"][ind] = ind_kps

        f.write(json.dumps(frame_record) + "\n")

file_size_mb = os.path.getsize(keypoints_path) / (1024 * 1024)
print(f"Wrote {len(df)} frame records to {keypoints_path} ({file_size_mb:.1f} MB)")

## 10. Upload Results to GCS

In [ ]:
gcs_keypoints_path = f"results/{VIDEO_ID}/keypoints.jsonl"
blob = bucket.blob(gcs_keypoints_path)
blob.upload_from_filename(keypoints_path)
print(f"Uploaded {keypoints_path} -> gs://{BUCKET}/{gcs_keypoints_path}")
print("\nDone! SuperAnimal evaluation complete.")